# Terrestrial analogs for lunar and Martian landing sites

This notebook screens a region of Earth for sites that resemble lunar and Martian reference sites. It scores each tile on morphometry, surface spectra and climate, then ranks the tiles.

**Two modes**
- `USE_SYNTHETIC = True` (default): runs fully offline on synthetic terrain. Use it to check your environment.
- `USE_SYNTHETIC = False`: live data from Microsoft Planetary Computer (Copernicus DEM, Sentinel-2) and NASA POWER. Requires network access and built reference descriptors (`python build_references.py reference_sites.json`).

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()                      # run the notebook from the project folder
sys.path.insert(0, str(ROOT))

from analog_finder.pipeline import run_search
from analog_finder.outputs import export_results, write_map

cfg = json.loads((ROOT / "config.json").read_text())
USE_SYNTHETIC = True                   # set to False for live data
pd.set_option("display.width", 160)
cfg

## 1. Reference sites
The reference descriptors are either synthetic (offline) or loaded from `reference_sites.json`.

In [ ]:
if USE_SYNTHETIC:
    from analog_finder.synthetic import make_synthetic_provider, synthetic_reference_sites
    refs = synthetic_reference_sites()
    provider = make_synthetic_provider()
else:
    from analog_finder.references import load_reference_sites
    from analog_finder.ingest import make_live_provider
    refs = load_reference_sites(ROOT / "reference_sites.json")
    provider = make_live_provider(cfg["date_range"], cfg["max_cloud_cover"])

pd.DataFrame([{k: r[k] for k in ("name", "body")} for r in refs])

## 2. Screen the region
Each tile is scored against every reference. The best-matching reference is reported per tile.

In [ ]:
df, descriptors = run_search(cfg, refs, provider)
print(f"{len(df)} tiles scored")
df.head(cfg["top_n"])

## 3. Top candidates and their characterization

In [ ]:
cols = ["tile_id", "center_lat", "center_lon", "composite_score", "score_morphometric",
        "score_spectral", "score_climate", "best_reference",
        "median_slope_deg", "roughness_m", "relief_m", "iron_oxide_ratio", "clay_ratio", "ndvi"]
top = df[[c for c in cols if c in df.columns]].head(cfg["top_n"])
top

## 4. Map

In [ ]:
out_dir = Path(cfg["output_dir"])
map_path = write_map(df, out_dir / "candidates_map.html")
if map_path:
    print("Map saved to", map_path)
    try:
        import folium   # noqa: F401
        from IPython.display import IFrame
        IFrame(str(map_path), width="100%", height=480)
    except Exception:
        pass
else:
    print("Folium is not installed; skipping the map (pip install folium)")

## 5. Export
CSV for spreadsheets, GeoJSON for GIS, and a JSON characterization summary for the top candidates.

In [ ]:
export_results(df, descriptors, out_dir, cfg["top_n"])
sorted(p.name for p in out_dir.iterdir())